# Tuning Veloce della Riduzione Dimensionale + Clustering

Questo notebook permette di eseguire la riduzione dimensionale su una matrice di input con parametri fissati, per poi esplorare ed eseguire il tuning degli algoritmi di clustering (K-Means, GMM, Agglomerative, Spectral, HDBSCAN) calcolando metriche geometriche e di stabilità (Consensus Clustering).

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tempfile
from IPython.display import Image, display

# Aggiunge la directory root del progetto al path per importare src
root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

# Ripristina il backend interattivo per i plot inline nel notebook
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

from src.utils.artifacts import load_matrix
from src.analysis.reduction import REDUCTION_METHODS, embedding_for_viz
from src.analysis.clustering import CLUSTERING_METHODS
from src.analysis.clustering_tuning import (
    run_clustering_tuning_sweep,
    compute_dendrogram_linkage,
    compute_eigengap,
    compute_silhouette_samples,
    consensus_suggestion_lines,
    METHOD_METRIC_COLUMNS,
    CONSENSUS_METRIC_COLUMNS
)
from src.analysis.plotting import (
    plot_clustering_tuning_metrics,
    plot_clustering_tuning_heatmaps,
    plot_dendrogram,
    plot_eigengap,
    plot_silhouette_analysis,
    plot_clusters_2d
)
from src.analysis.params import load_method_params, load_tuning_grid, load_consensus_config

print("Librerie importate con successo e backend matplotlib impostato per visualizzazione inline.")

## 1. Caricamento Dati ed Esecuzione Riduzione Dimensionale (Fissata)

Carichiamo la matrice lesionale e calcoliamo l'embedding di riferimento usando parametri di produzione predefiniti (es. UMAP con metric = 'jaccard' ed n_neighbors = 15).

In [ ]:
# Percorso della matrice lesionale di input (modificabile all'occorrenza)
input_matrix_dir = Path("../data/derived/lesion_matrix/21-07_s1.1")

try:
    X, metadata, extra_arrays = load_matrix(input_matrix_dir)
    print(f"Matrice caricata correttamente. Shape di X: {X.shape[0]} soggetti x {X.shape[1]} features")
except Exception as e:
    print(f"Errore nel caricamento della matrice: {e}")

# Configuriamo ed eseguiamo UMAP con i parametri di produzione fissati
reduction_method = "umap"
params_file = Path("../config/registry/params_reduction.json")

# Carichiamo i parametri di default dal registro
reduction_params, _ = load_method_params(params_file, reduction_method)
# Fissiamo i parametri nested per la nostra analisi veloce
reduction_params["metric"] = "jaccard"
reduction_params["regress_out_volume"] = False
reduction_params["n_neighbors"] = 15
reduction_params["min_dist"] = 0.0

print(f"Esecuzione riduzione dimensionale ({reduction_method.upper()}) con parametri: {reduction_params}")
embedding = REDUCTION_METHODS[reduction_method](X, reduction_params)
print(f"Shape dell'embedding risultante: {embedding.shape}")

# Embedding a 2 dimensioni per visualizzazioni statiche
viz_embedding = embedding_for_viz(reduction_method, X, reduction_params, embedding, 2)

## 2. Tuning di un Algoritmo di Clustering

Scegliamo un algoritmo di clustering e definiamo la griglia dei parametri da esplorare. Definiamo inoltre se abilitare le metriche di stabilità (Consensus Clustering) per supportare la scelta del numero ottimo di cluster.

In [ ]:
# Scegli l'algoritmo di clustering da accordare: "kmeans", "agglomerative", "gmm", "spectral", "hdbscan"
clustering_method = "kmeans"
clustering_params_file = Path("../config/registry/params_clustering.json")

# Carica parametri di base e griglia di tuning dal registro
base_params, _ = load_method_params(clustering_params_file, clustering_method)
tuning_grid = load_tuning_grid(clustering_params_file, clustering_method)
consensus_config = load_consensus_config(clustering_params_file, clustering_method)

# Possiamo forzare/modificare la griglia di sweep e la stabilità
# Ad esempio per K-Means: sweeps n_clusters da 2 a 10
if clustering_method in ["kmeans", "agglomerative", "spectral", "gmm"]:
    tuning_grid = {"n_clusters" if clustering_method != "gmm" else "n_components": list(range(2, 11))}

# Riduciamo il numero di repeats di stabilità per rendere la run interattiva veloce (es. 20 ripetizioni)
if consensus_config:
    if "rsc" in consensus_config:
        consensus_config["rsc"]["n_repeats"] = 20  
    if "monti" in consensus_config:
        consensus_config["monti"]["n_repeats"] = 20  

print(f"Algoritmo: {clustering_method}")
print(f"Base params: {base_params}")
print(f"Grid di tuning: {tuning_grid}")
print(f"Consensus config: {consensus_config}")

## 3. Esecuzione del Tuning Sweep

Eseguiamo il tuning sweep che calcolerà Silhouette, Calinski-Harabasz, Davies-Bouldin, stabilità Monti e RSC Eigengap.

In [ ]:
print(f"Esecuzione dello sweep di tuning per {clustering_method}...")
results = run_clustering_tuning_sweep(
    clustering_method, embedding, base_params, tuning_grid, consensus_config
)

# Mostra i risultati
display(results)

## 4. Visualizzazione delle Metriche di Tuning

Mostriamo i grafici delle metriche per individuare il numero ottimale di cluster.
- Se varia **un solo parametro**, mostriamo l'andamento delle metriche in curve sovrapposte.
- Se variano **due parametri**, mostriamo le heatmaps delle metriche.

In [ ]:
swept_params = list(tuning_grid.keys())
metric_cols = METHOD_METRIC_COLUMNS[clustering_method] + [c for c in CONSENSUS_METRIC_COLUMNS if c in results.columns]

# Generiamo i suggerimenti letterari basati su Consensus (se presenti)
suggestions = consensus_suggestion_lines(results, clustering_method)
if suggestions:
    print("Suggerimenti stabilità / consensus:")
    for sug in suggestions:
        print(f" - {sug}")

with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmpfile:
    tmp_path = Path(tmpfile.name)

try:
    title = f"Tuning {clustering_method} on {reduction_method.upper()}"
    if len(swept_params) == 1:
        plot_clustering_tuning_metrics(results, swept_params[0], metric_cols, tmp_path, title)
        display(Image(filename=tmp_path))
    elif len(swept_params) == 2:
        plot_clustering_tuning_heatmaps(results, swept_params[0], swept_params[1], metric_cols, tmp_path, title)
        display(Image(filename=tmp_path))
    else:
        print(f"Troppi parametri swept ({len(swept_params)}), plot non supportato. Guarda i dati sopra.")
finally:
    if tmp_path.exists():
        tmp_path.unlink()

## 5. Visualizzazione Diagnostica Standalone (Dendrogramma o Eigengap)

Per i metodi `agglomerative` e `spectral`, la pipeline calcola diagnostiche aggiuntive indipendenti dal taglio di cluster:
- **Dendrogramma** (Agglomerative)
- **Eigengap plot** (Spectral)

In [ ]:
if clustering_method == "agglomerative":
    print("Calcolo e plotting del dendrogramma...")
    linkage_matrix = compute_dendrogram_linkage(embedding, base_params)
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmpfile:
        tmp_path = Path(tmpfile.name)
    try:
        plot_dendrogram(linkage_matrix, tmp_path, f"Dendrogram - {clustering_method}")
        display(Image(filename=tmp_path))
    finally:
        if tmp_path.exists():
            tmp_path.unlink()
            
elif clustering_method == "spectral":
    print("Calcolo e plotting dell'eigengap graph...")
    eigenvalues = compute_eigengap(embedding, base_params, max_k=20)
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmpfile:
        tmp_path = Path(tmpfile.name)
    try:
        plot_eigengap(eigenvalues, tmp_path, f"Eigengap Graph - {clustering_method}")
        display(Image(filename=tmp_path))
    finally:
        if tmp_path.exists():
            tmp_path.unlink()
else:
    print(f"Nessuna diagnostica standalone per il metodo {clustering_method}.")

## 6. Visualizzazione Spaziale delle Partizioni ed Analisi Silhouette

Selezioniamo una configurazione ottimale ed eseguiamo il fit finale per visualizzare la partizione e il diagramma silhouette per-soggetto.

In [ ]:
# Scegli la configurazione ottimale da ispezionare visivamente
n_clusters_opt = 4
best_params = {**base_params}

param_key = "n_components" if clustering_method == "gmm" else "n_clusters"
if param_key in best_params or clustering_method in ["kmeans", "agglomerative", "spectral", "gmm"]:
    best_params[param_key] = n_clusters_opt

print(f"Esecuzione fit finale con parametri: {best_params}")
labels = CLUSTERING_METHODS[clustering_method](embedding, best_params)

# 1. Visualizziamo i cluster nello spazio 2D dell'embedding
with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmpfile:
    tmp_path = Path(tmpfile.name)
try:
    plot_clusters_2d(
        viz_embedding, 
        labels, 
        tmp_path, 
        f"{reduction_method} dim 1", 
        f"{reduction_method} dim 2", 
        f"Partition {clustering_method.capitalize()} (k={n_clusters_opt})"
    )
    display(Image(filename=tmp_path))
finally:
    if tmp_path.exists():
        tmp_path.unlink()

# 2. Analisi Silhouette per-soggetto
try:
    sample_labels, sample_silhouette_values = compute_silhouette_samples(embedding, labels)
    
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmpfile:
        tmp_path = Path(tmpfile.name)
        
    try:
        plot_silhouette_analysis(
            sample_labels,
            sample_silhouette_values,
            viz_embedding,
            labels,
            tmp_path,
            f"{reduction_method} dim 1",
            f"{reduction_method} dim 2",
            f"Silhouette Analysis - {clustering_method.capitalize()} (k={n_clusters_opt})"
        )
        display(Image(filename=tmp_path))
    finally:
        if tmp_path.exists():
            tmp_path.unlink()
except Exception as e:
    print(f"Impossibile eseguire l'analisi silhouette per questa configurazione: {e}")